# Simulación Avanzada

## Ejercicio 1.8 B.Calderón. Introducción a la Simulación

Kevin Ferney Hidalgo Higuita
kfhidalgoh@unal.edu.co
Repositorio [GitHub](https://github.com/thepadr30/SimulAva3010192)

### Librerías y configuraciones iniciales

In [ ]:
import logging
import os
import sys
import warnings
from time import localtime, strftime

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML
from IPython.display import Image, display

from src.graph.fun_graph_matplotlib import FnGraphMat
from src.logs.logger import setup_logging
from src.utils import statisticsBase as sB
from src.ipynb.estiloDashboard import estilo_dashboard, estilo_dashboard_v0, estilo_dashboard_v2

In [ ]:
__authors__ = ["Kevin Hidalgo"]
__contact__ = "kfhidalgoh@unal.edu.co"
__copyright__ = "Copyright 2026"
__credits__ = ["Kevin Hidalgo"]
__email__ = "kfhidalgoh@unal.edu.co"
__status__ = "Desarrollo"
__version__ = "1.0.0"
__date__ = "2026-03-11"
__file__ = "KevinHidalgoTarea3SimulacionA2026Ipynb"

file_log = os.path.join('D:\SimulAva\logs', strftime("%Y%m%d%H%M%S", localtime()) + "_" + __file__ + ".log" )

logger = setup_logging(file_log)
logging.info("Python %s on %s", sys.version, sys.platform)
logging.info("Root: %s", os.getcwd())  # os.path.abspath(os.curdir)
logging.info("Log: %s", file_log)

##############
# Constantes #
##############

varData = r'D:\SimulAva\data\raw'
grmt = FnGraphMat('seaborn-v0_8-darkgrid')  # seaborn-v0_8-darkgrid, dark_style
# sns.set_palette("husl")
warnings.filterwarnings('ignore')
np.random.seed(2026)
%matplotlib inline

### Enunciado

Modifique el diagrama de flujo y el programa del juego de la moneda para
analizar el mismo juego de la moneda con las siguientes modificaciones: El
juego termina cuando la diferencia entre caras y sellos sea 3 ó cuando el
número de lanzamientos realizados sea 15.

Primero vamos a entender el juego cuando la diferencia entre cara y sello sea 3

### Las reglas del juego son estas:

1. **Lanzamientos**
2. **Cuándo termina**
3. **Pagos y cobros**
    * Premio fijo de $\$8.00$
    * Cada lanzamiento debe pagar $\$1.00$

*¿Te conviene participar en este juego?*

Tenemos 3 opciones:

* Matemática
* Jugando en la vida real
* Simulación

#### Solución matemática

* **La fórmula del dinero**: La utilidad (lo que ganas o pierdes) se calcula de la siguiente forma $U\left(x\right)=Ingresos-Egresos=8-x\quad x=3,5,7,\ldots$,  donde X es el número de lanzamientos.
* **Sólo números impares**: *nunca puede terminar en un número par*.
* **El objetivo**: Calcular la *utilidad esperada* $E\left(U\right)=\sum U\left(x\right)P\left(x\right)$, donde $P\left(x\right)$ es la función de densidad del número de lanzamientos en que puede terminar el juego, y representa la probabilidad de que el juego termine en $x$ lanzamientos.
* **La dificultad**: debemos conocer $P\left(x\right)$ para los diferentes valores de $x$.

\[
P\left(X=3\right)=\begin{cases}
Probabilidad\:de\:que\:el\:juego\:termine\:en\:tres\:lanzamientos\\
Probabilidad\:de\:que\:los\:tres\:primeros\:lanzamientos\:sean\:todos\:cara\:\acute{o}\:todos\:sean\:sellos\\
Prob(tres\:primeros\:lanzamientos\:sean\:cara)+Prob(tres\:primeros\:lanzamientos\:sean\:sello)\\
P\left(CCC\right)+P\left(SSS\right)=\left(\frac{1}{2}\right)^{3}+\left(\frac{1}{2}\right)^{3}=\frac{1}{4}
\end{cases}
\]

\[
P\left(X=5\right)=\begin{cases}
Probabilidad\:de\:que\:el\:juego\:termine\:en\:5\:lanzamientos\\
Probabilidad\:de\:obtener\:cuatro\:caras\:y\:un\:sello\:\acute{o}\:cuatro\:sellos\:y\:una\:cara,\:dado\:que\:el\:juego\:no\:hab\acute{\imath}a\:terminado\:antes\\
P\left(CCSCC\right)+P\left(CSCCC\right)+P\left(SCCCC\right)+P\left(SSCSS\right)+P\left(SCSSS\right)+P\left(CSSSS\right)=6\left(\frac{1}{2}\right)^{5}=\frac{3}{16}
\end{cases}
\]

Aunque es posible crear un modelo matemático con fórmulas, llegar a una solución final por este camino es bastante complicado, incluso para un juego tan simple como este.

#### Experimentar con el sistema real

* **Definir la muestra**
* **El proceso de cada juego**:
    * Contador de "Caras" y "Sellos" en *cero*.
    * Lanzar una moneda y anotar el resultado.
    * **Regla de parada**
* **Calcular la ganancia**
* **El promedio final**

#### Solución por simulador

La simulación consiste en imitar el juego real, pero sin necesidad de tener una moneda física en la mano ni de lanzar un brazo mil veces.

* **Preparación**.
* **Inicio de cada juego**.
* **El "lanzamiento" virtual**.
* **Conteo y parada**.
* **Resultado económico**.
* **El gran promedio**

```mermaid
graph TD
    %% Definición de Estilos Académicos
    classDef terminal fill:#37474f,stroke:#263238,color:#fff,stroke-width:2px
    classDef logic fill:#ffffff,stroke:#1e88e5,color:#0d47a1,stroke-width:2px
    classDef decision fill:#ffffff,stroke:#fbc02d,color:#7f6000,stroke-width:2px
    classDef data fill:#f1f8e9,stroke:#558b2f,color:#2e7d32,stroke-width:1px,stroke-dasharray: 5 5

    %% Nodos Principales
    Start((Inicio)):::terminal --> Param[1. Definición de Parámetros del Sistema]:::logic

    subgraph SIM [Ciclo de Réplicas de Simulación]
        Param --> Init[2. Inicialización de Acumuladores de Estado]:::logic
        Init --> Loop{3. Control de<br/>Convergencia}:::decision

        subgraph CORE [Modelo Estocástico]
            Loop -- "Próximo Juego" --> Game[4. Generación de Trayectoria Estocástica]:::logic
            Game --> Eval["5. Determinación del Punto de Parada |C-S|=3"]:::logic
        end

        Eval --> Update[6. Actualización de Estadísticos de Desempeño]:::logic
        Update --> Freq[7. Clasificación en Distribución de Frecuencias]:::logic
        Freq --> Increment[8. Incremento de Contador de Juegos]:::logic
        Increment --> Loop
    end

    Loop -- "Fin de Simulación" --> Stats[9. Estimación de Parámetros Poblacionales]:::logic

    subgraph ANALY [Módulo de Análisis y Salida]
        Stats --> Var[10. Análisis de Variabilidad y Dispersión]:::logic
        Var --> Report[/11. Reporte de Resultados y Distribución Relativa/]:::data
    end

    Report --> End((Fin)):::terminal

    %% Notas de Identificación de Estilo
    subgraph Leyenda
        L1[Proceso Lógico]:::logic
        L2{Punto de Decisión}:::decision
        L3[/Salida de Datos/]:::data
    end
```

Para lograr una estética de nivel académico (tipo *Discrete-Event System Simulation* de Law & Kelton), debemos alejarnos del detalle técnico algorítmico y enfocarnos en la **arquitectura lógica** de la simulación.

Esta versión sintetiza el proceso en exactamente **12 nodos estratégicos**, utilizando una paleta de colores sobria (azules acero, grises pizarra y blancos) y terminología técnica precisa.

**Desglose de los 12 Nodos (Nivel Tesis)**

1. **Definición de Parámetros:** Entrada de variables exógenas ($P(cara)$, $N$, semillas).
2. **Inicialización:** Configuración del estado inicial del sistema ($X_{min}, X_{max}, \sum X$).
3. **Control de Convergencia:** Nodo de decisión que gobierna el número de réplicas ($n \leq NRO\_JUEGOS$).
4. **Modelo Estocástico:** Núcleo de la simulación donde ocurre la generación de variables aleatorias.
5. **Punto de Parada:** Lógica interna que define el fin de un evento (la diferencia crítica de 3).
6. **Actualización de Estadísticos:** Captura de métricas de desempeño por juego ($LANZA$).
7. **Clasificación:** Mapeo de resultados continuos/discretos en intervalos de frecuencia.
8. **Avance de Simulación:** El reloj de simulación o contador de réplicas.
9. **Estimación de Parámetros:** Cálculo de la media de la muestra y utilidad esperada.
10. **Análisis de Variabilidad:** Cálculo de varianza y desviación estándar para determinar la precisión.
11. **Reporte de Resultados:** Generación de la tabla de frecuencias y resumen estadístico.
12. **Fin:** Cierre del proceso y liberación de memoria.

**Características de este Diseño**:

* **Abstracción:** Sustituye el "Caras = Caras + 1" por "Generación de Trayectoria Estocástica", lo cual es el lenguaje apropiado para artículos científicos.
* **Modularidad:** Los `subgraph` separan claramente la configuración, la ejecución y el análisis post-simulación.
* **Simetría:** El flujo es lineal y descendente, facilitando su inserción en columnas de texto de una tesis o presentación de PowerPoint.


```mermaid
flowchart TD
    %% Definición de estilos
    classDef input fill:#e8f0fe,stroke:#1a73e8,color:#174ea6,stroke-width:2px
    classDef init fill:#e6f7e6,stroke:#34a853,color:#0d652d,stroke-width:2px
    classDef process fill:#fff3e0,stroke:#f9ab00,color:#b45f06,stroke-width:2px
    classDef loop fill:#fce8e6,stroke:#d93025,color:#a50e0e,stroke-width:2px
    classDef condition fill:#f3e5f5,stroke:#9334e6,color:#4a148c,stroke-width:2px
    classDef output fill:#e6f7ff,stroke:#039be5,color:#01579b,stroke-width:2px
    classDef child fill:#f5f5f5,stroke:#9e9e9e,color:#424242,stroke-dasharray: 5 5

    subgraph INPUT["📥 Input Module"]
        direction TB
        A([Inicio]) --> B[Leer: SEMILLA, NRO_JUEGOS,<br>PROB_CARA, Xo, ΔX, NRO_INT]
    end

    subgraph INIT["⚙️ Initialization"]
        direction TB
        C["Inicializar variables:<br>ΣX = 0<br>ΣX² = 0<br>Xmin = 100<br>Xmax = 0<br>ΣU = 0<br>FRECUENCIA[INT] = 0<br>JUEGOS = 1"]
    end

    subgraph SIM["🎲 Simulation Engine"]
        direction TB
        D{¿JUEGOS ≤ NRO_JUEGOS?}

        subgraph GAME["🎯 Individual Game"]
            direction TB
            E[Inicializar juego:<br>CARAS = 0<br>SELLOS = 0]

            subgraph COIN["🪙 Coin Toss Loop"]
                direction TB
                F{"¿|CARAS - SELLOS| ≠ 3?"}
                G[Generar número aleatorio R]
                H{¿R < PROB_CARA?}
                I[CARAS = CARAS + 1]
                J[SELLOS = SELLOS + 1]
            end

            K[LANZA = CARAS + SELLOS]
            L[ΣX = ΣX + LANZA]
            M[ΣX² = ΣX² + LANZA²]

            subgraph EXTREMES["📊 Update Extremes"]
                direction TB
                N{¿LANZA < Xmin?}
                O[Xmin = LANZA]
                P{¿LANZA > Xmax?}
                Q[Xmax = LANZA]
            end

            subgraph INTERVAL["📈 Interval Calculation"]
                direction TB
                R["Calcular INT:<br>INT = trunc((LANZA - Xo) / ΔX) + 2"]
                S{¿INT < 1?}
                T[INT = 1]
                U{¿INT > NRO_INT?}
                V[INT = NRO_INT]
                W["FRECUENCIA[INT] = FRECUENCIA[INT] + 1"]
            end

            X[JUEGOS = JUEGOS + 1]
        end

        %% Conexiones dentro de Simulation Engine
        D -- Sí --> E
        E --> F
        F -- Sí --> G
        G --> H
        H -- Sí --> I
        H -- No --> J
        I --> F
        J --> F
        F -- No --> K
        K --> L
        L --> M
        M --> N
        N -- Sí --> O
        N -- No --> P
        O --> P
        P -- Sí --> Q
        P -- No --> R
        Q --> R
        R --> S
        S -- Sí --> T
        S -- No --> U
        T --> U
        U -- Sí --> V
        U -- No --> W
        V --> W
        W --> X
        X --> D
    end

    subgraph STATS["📊 Statistical Processing"]
        direction TB
        Y["Calcular estadísticas finales:<br>MEDIA = ΣX / NRO_JUEGOS<br>VARIANZA = (ΣX² - NRO_JUEGOS * MEDIA²) / (NRO_JUEGOS - 1)<br>DESVIACION = sqrt(VARIANZA)<br>UTILIDAD_MEDIA = ΣU / NRO_JUEGOS"]
    end

    subgraph OUTPUT["📋 Output / Reporting"]
        direction TB
        Z["Imprimir resultados generales:<br>'NRO_JUEGOS, MEDIA,<br>DESVIACION, Xmin, Xmax,<br>UTILIDAD_MEDIA'"]

        subgraph FREQ["📊 Frequency Report"]
            direction TB
            AA[INTERV = 1]
            AB{¿INTERV ≤ NRO_INT?}
            AC["FRECUENCIA_RELATIVA =<br>FRECUENCIA[INTERV] * 100 / NRO_JUEGOS"]
            AD["Imprimir intervalo:<br>'INTERV, FRECUENCIA[INTERV],<br>FRECUENCIA_RELATIVA'"]
            AE[INTERV = INTERV + 1]
        end

        AF([Fin Simulación])
    end

    %% Conexiones entre módulos principales
    B --> C
    C --> D
    D -- No --> Y
    Y --> Z
    Z --> AA
    AA --> AB
    AB -- Sí --> AC
    AC --> AD
    AD --> AE
    AE --> AB
    AB -- No --> AF

    %% Asignación de clases
    class A,B input
    class C init
    class D,F,H,N,P,S,U,AB condition
    class E,G,I,J,K,L,M,O,Q,R,T,V,W,X,AC,AD,AE process
    class COIN,GAME,EXTREMES,INTERVAL,FREQ loop
    class Y,Z,AF output
    class INPUT,INIT,SIM,STATS,OUTPUT child
```

INPUT: Módulo de entrada de datos
INIT: Módulo de inicialización de variables
SIM: Motor de simulación principal
GAME: Submódulo para cada juego individual
COIN: Bucle de lanzamiento de moneda
EXTREMES: Actualización de valores extremos
INTERVAL: Cálculo de intervalos de frecuencia
STATS: Procesamiento estadístico
OUTPUT: Salida y reportes
FREQ: Reporte de frecuencias por intervalo

ClassDef con semántica mejorada:
input: Para operaciones de entrada (verde azulado)
init: Para inicializaciones (verde)
process: Para operaciones de proceso (naranja)
loop: Para estructuras de bucle (rojo claro)
condition: Para decisiones/condicionales (púrpura)
output: Para salidas de datos (azul claro)
child: Para subgraphs (gris punteado)